# Valuation-to-Option Pricing Lab

Two tracks remain distinct: a belief-based analyst-scenario payoff and a CRR no-arbitrage value.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')


In [1]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}; weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({'state':list(quantile_map),'quantile':list(quantile_map.values()),'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],'subjective_weight':[weights[k] for k in quantile_map]})
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float); return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type']); states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum()); discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years'])); belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum()); weight_zero=1-weight_positive


In [1]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]; conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_disp=float(np.log(targets.target/M['spot']).std(ddof=1)); crr_disp=float(M['volatility']*np.sqrt(M['maturity_years']))
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
results={'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},'distribution_comparison':{'analyst_log_dispersion':analyst_disp,'crr_horizon_volatility':crr_disp,'message':'Analyst targets are narrower than the volatility input used by CRR.' if analyst_disp<crr_disp else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'},'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']},'analyst_horizon_years':float(targets.horizon_months.iloc[0]/12)}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2)); results['comparison']


## Decision memo prompt

Explain the valuation evidence, distribution, payoff-producing states, model risk, conclusion, and a numerical reversal condition.